# DDR Training — Variant A (baseline) and Variant B (CORAL ordinal loss)

Runs on Colab's free GPU instead of locally, since DDR (12,522 images) is too large for reliable local training on an M3 Air (two local attempts hit severe memory/swap issues — see DEVELOPMENT_LOG.md).

**Before running:**
1. Runtime > Change runtime type > GPU (T4 is fine).
2. Set up Kaggle credentials as Colab Secrets (does NOT expose them in this notebook, which lives in a public repo): click the key icon (🔑) in the left sidebar, add two secrets named `KAGGLE_USERNAME` and `KAGGLE_KEY` with your Kaggle account's values, and toggle notebook access on for both.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/Dilshan-Fernando-01/Computer-Vision-Assignment.git
%cd Computer-Vision-Assignment

In [ ]:
!pip install -q -r requirements.txt kaggle

In [ ]:
import os
from google.colab import userdata

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    f.write('{"username":"%s","key":"%s"}' % (userdata.get('KAGGLE_USERNAME'), userdata.get('KAGGLE_KEY')))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle credentials written (not printed, not committed - Colab secrets stay local to your account)')

In [ ]:
!python3 scripts/download_ddr_dataset.py

In [ ]:
!python3 src/datasets/splits.py

## Variant A — baseline (EfficientNet-B0, plain cross-entropy)

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

from src.datasets.dataset import DRGradingDataset
from src.augmentation.augment import get_training_augmentations, make_sample_weights
from src.models.build_model import build_model
from src.training.train import fit, evaluate, get_device

DEVICE = get_device()
IMAGE_SIZE = 512
BATCH_SIZE = 32
EPOCHS = 40
PATIENCE = 8
LR = 1e-4

print('device:', DEVICE)

train_ds = DRGradingDataset('data/processed/splits/train.csv', image_size=IMAGE_SIZE, transform=get_training_augmentations(IMAGE_SIZE))
val_ds   = DRGradingDataset('data/processed/splits/val.csv',   image_size=IMAGE_SIZE)
test_ds  = DRGradingDataset('data/processed/splits/test.csv',  image_size=IMAGE_SIZE)

train_labels = [label for _, label in train_ds.samples]
sample_weights = make_sample_weights(train_labels)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, num_workers=2)

print('train/val/test sizes:', len(train_ds), len(val_ds), len(test_ds))

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
model_a = build_model('efficientnet_b0', num_classes=5, pretrained=True)

history_a = fit(
    model_a, train_loader, val_loader, criterion,
    epochs=EPOCHS, lr=LR, patience=PATIENCE,
    checkpoint_path='outputs/checkpoints/variant_a_efficientnet_b0.pt',
    device=DEVICE,
)

In [ ]:
import json, os
os.makedirs('outputs/history', exist_ok=True)
with open('outputs/history/variant_a_history.json', 'w') as f:
    json.dump(history_a, f, indent=2)
print('best val macro F1:', history_a['best_val_macro_f1'])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history_a['train_loss'], label='train'); axes[0].plot(history_a['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history_a['train_acc'], label='train'); axes[1].plot(history_a['val_acc'], label='val')
axes[1].set_title('Accuracy'); axes[1].legend()
axes[2].plot(history_a['val_macro_f1'], color='green'); axes[2].set_title('Val Macro F1')
fig.suptitle('Variant A - DDR')
fig.tight_layout()
fig.savefig('outputs/history/variant_a_curves.png', dpi=120)
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model_a.load_state_dict(torch.load('outputs/checkpoints/variant_a_efficientnet_b0.pt', map_location=DEVICE))
test_metrics_a = evaluate(model_a, test_loader, criterion, DEVICE)

print('Test accuracy:', test_metrics_a['accuracy'])
print('Test macro F1:', test_metrics_a['macro_f1'])
print(classification_report(test_metrics_a['labels'], test_metrics_a['preds'], target_names=[f'Stage {i}' for i in range(5)]))
print(np.array(confusion_matrix(test_metrics_a['labels'], test_metrics_a['preds'])))

## Variant B — CORAL ordinal loss

In [ ]:
from src.models.ordinal import CoralModel, CoralCriterion, coral_predict

criterion_b = CoralCriterion(num_classes=5)
model_b = CoralModel('efficientnet_b0', num_classes=5, pretrained=True)

history_b = fit(
    model_b, train_loader, val_loader, criterion_b,
    epochs=EPOCHS, lr=LR, patience=PATIENCE,
    checkpoint_path='outputs/checkpoints/variant_b_coral_efficientnet_b0.pt',
    device=DEVICE,
    predict_fn=coral_predict,
)

In [ ]:
with open('outputs/history/variant_b_history.json', 'w') as f:
    json.dump(history_b, f, indent=2)

model_b.load_state_dict(torch.load('outputs/checkpoints/variant_b_coral_efficientnet_b0.pt', map_location=DEVICE))
test_metrics_b = evaluate(model_b, test_loader, criterion_b, DEVICE, predict_fn=coral_predict)

print('Test accuracy:', test_metrics_b['accuracy'])
print('Test macro F1:', test_metrics_b['macro_f1'])
print(classification_report(test_metrics_b['labels'], test_metrics_b['preds'], target_names=[f'Stage {i}' for i in range(5)]))
print(np.array(confusion_matrix(test_metrics_b['labels'], test_metrics_b['preds'])))

## Download results back to your machine

Run this last, then drag the downloaded `outputs.zip` into your local `outputs/` folder (or just re-tell Claude the printed numbers above and it'll update the docs).

In [ ]:
!zip -r outputs.zip outputs/checkpoints outputs/history
from google.colab import files
files.download('outputs.zip')